In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import numpy as np
from datetime import datetime
import os


In [4]:
HEADERS = {"User-Agent": "Mozilla/5.0"}


In [5]:
# Create the links file inside Colab
links_content = """
https://www.globalfirepower.com/countries-listing.php
https://www.globalfirepower.com/total-population-by-country.php
https://www.globalfirepower.com/available-military-manpower.php
https://www.globalfirepower.com/manpower-fit-for-military-service.php
https://www.globalfirepower.com/manpower-reaching-military-age-annually.php
https://www.globalfirepower.com/active-military-manpower.php
https://www.globalfirepower.com/active-reserve-military-manpower.php
https://www.globalfirepower.com/manpower-paramilitary.php
https://www.globalfirepower.com/aircraft-total.php
https://www.globalfirepower.com/aircraft-total-fighters.php
https://www.globalfirepower.com/aircraft-total-attack-types.php
https://www.globalfirepower.com/aircraft-total-transports.php
https://www.globalfirepower.com/aircraft-total-trainers.php
https://www.globalfirepower.com/aircraft-total-special-mission.php
https://www.globalfirepower.com/aircraft-total-tanker-fleet.php
https://www.globalfirepower.com/aircraft-helicopters-total.php
https://www.globalfirepower.com/aircraft-helicopters-attack.php
https://www.globalfirepower.com/armor-tanks-total.php
https://www.globalfirepower.com/armor-apc-total.php
https://www.globalfirepower.com/armor-self-propelled-guns-total.php
https://www.globalfirepower.com/armor-towed-artillery-total.php
https://www.globalfirepower.com/armor-mlrs-total.php
https://www.globalfirepower.com/navy-ships.php
https://www.globalfirepower.com/navy-force-by-tonnage.php
https://www.globalfirepower.com/navy-aircraft-carriers.php
https://www.globalfirepower.com/navy-helo-carriers.php
https://www.globalfirepower.com/navy-submarines.php
https://www.globalfirepower.com/navy-destroyers.php
https://www.globalfirepower.com/navy-frigates.php
https://www.globalfirepower.com/navy-corvettes.php
https://www.globalfirepower.com/navy-patrol-coastal-craft.php
https://www.globalfirepower.com/navy-mine-warfare-craft.php
https://www.globalfirepower.com/defense-spending-budget.php
https://www.globalfirepower.com/external-debt-by-country.php
https://www.globalfirepower.com/purchasing-power-parity.php
https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php
https://www.globalfirepower.com/major-serviceable-airports-by-country.php
https://www.globalfirepower.com/labor-force-by-country.php
https://www.globalfirepower.com/major-ports-and-terminals.php
https://www.globalfirepower.com/merchant-marine-strength-by-country.php
https://www.globalfirepower.com/railway-coverage.php
https://www.globalfirepower.com/roadway-coverage.php
https://www.globalfirepower.com/oil-production-by-country.php
https://www.globalfirepower.com/oil-consumption-by-country.php
https://www.globalfirepower.com/proven-oil-reserves-by-country.php
https://www.globalfirepower.com/natural-gas-production-by-country.php
https://www.globalfirepower.com/natural-gas-consumption-by-country.php
https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php
https://www.globalfirepower.com/coal-production-by-country.php
https://www.globalfirepower.com/coal-consumption-by-country.php
https://www.globalfirepower.com/proven-coal-reserves-by-country.php
https://www.globalfirepower.com/square-land-area.php
https://www.globalfirepower.com/coastline-coverage.php
https://www.globalfirepower.com/border-coverage.php
https://www.globalfirepower.com/waterway-coverage.php
"""

with open("links_for_military_data.txt", "w") as f:
    f.write(links_content.strip())


In [6]:
def read_links_txt(path="links_for_military_data.txt"):
    links = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and line.startswith("http"):
                links.append(line)
    return links


In [7]:
def scrape_base_countries():
    url = "https://www.globalfirepower.com/countries-listing.php"
    r = requests.get(url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(r.text, "html.parser")

    containers = soup.find_all("div", class_="picTrans recordsetContainer boxShadow zoom")
    countries, ranks = [], []

    for c in containers:
        try:
            country = c.find("span", class_="textWhite textLarge textShadow").text.strip()
            rank = c.find("span", class_="textWhite textLarge textBold").text.strip()
            countries.append(country)
            ranks.append(int(rank))
        except:
            continue

    df = pd.DataFrame({"country": countries, "rank": ranks})
    return df


In [8]:
def scrape_metric(url, column_name):
    r = requests.get(url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(r.text, "html.parser")

    containers = soup.find_all("div", class_="picTrans recordsetContainer boxShadow zoom")
    countries, values = [], []

    for c in containers:
        try:
            country = c.find("span", class_="textWhite textLarge textShadow").text.strip()
            value = c.find_all("span", class_="textWhite textLarge")[-1].text.strip()
            countries.append(country)
            values.append(value)
        except:
            continue

    df = pd.DataFrame({"country": countries, column_name: values})

    # Clean numeric values
    df[column_name] = df[column_name].apply(lambda x: float(re.sub(r"[^\d.-]", "", str(x))) if pd.notnull(x) else np.nan)
    return df


In [9]:
os.makedirs("data", exist_ok=True)

def run_scraping():
    df_main = scrape_base_countries()
    urls = read_links_txt()

    for url in urls:
        if "countries-listing.php" in url:
            continue
        column_name = url.split("/")[-1].replace(".php", "")
        df_metric = scrape_metric(url, column_name)
        df_main = df_main.merge(df_metric, on="country", how="left")

    return df_main

df_raw = run_scraping()
df_raw.to_csv("data/military_metrics_raw.csv", index=False)
df_raw.head()


,country,rank,total-population-by-country,available-military-manpower,manpower-fit-for-military-service,manpower-reaching-military-age-annually,active-military-manpower,active-reserve-military-manpower,manpower-paramilitary,aircraft-total,...,natural-gas-production-by-country,natural-gas-consumption-by-country,proven-natural-gas-reserves-by-country,coal-production-by-country,coal-consumption-by-country,proven-coal-reserves-by-country,square-land-area,coastline-coverage,border-coverage,waterway-coverage
0,United States,1,3.419634e+08,150463900.0,124816644.0,4445524.0,1333030.0,799500.0,0.0,13032.0,...,1.029000e+12,9.143010e+11,1.340200e+13,5.342340e+08,4.951560e+08,2.478830e+11,9833517.0,19924.0,12002.0,41009.0
1,Russia,2,1.408208e+08,69002197.0,46189226.0,1267387.0,1320000.0,2000000.0,250000.0,4237.0,...,6.178300e+11,4.722390e+11,4.780500e+13,5.311300e+08,2.907630e+08,1.621660e+11,17098242.0,37653.0,22407.0,102000.0
2,China,3,1.415043e+09,764123366.0,626864169.0,19810606.0,2035000.0,510000.0,625000.0,3529.0,...,2.253410e+11,3.661600e+11,6.654000e+12,4.805000e+09,5.191000e+09,1.570410e+11,9596960.0,14500.0,22457.0,27700.0
3,India,4,1.409128e+09,662290299.0,522786598.0,23955181.0,1431000.0,1000000.0,2527000.0,2183.0,...,3.317000e+10,5.886700e+10,1.381000e+12,1.020000e+09,1.262000e+09,1.277270e+11,3287263.0,7000.0,13888.0,14500.0
4,South Korea,5,5.208180e+07,26040900.0,21353538.0,416654.0,450000.0,3100000.0,120000.0,1540.0,...,5.512700e+07,5.948000e+10,7.079000e+09,1.608100e+07,1.368170e+08,3.260000e+08,99720.0,2413.0,237.0,1600.0


## **Data Cleaning**

In [10]:
# Static reference for mandatory columns
country_meta = {
    "India":       ("New Delhi", "South Asia", "Asia", "Non-NATO"),
    "United States": ("Washington, D.C.", "North America", "North America", "NATO"),
    "China":       ("Beijing", "East Asia", "Asia", "Non-NATO"),
    "Russia":      ("Moscow", "Eastern Europe", "Europe", "Non-NATO"),
    "United Kingdom": ("London", "Western Europe", "Europe", "NATO"),
    "France":      ("Paris", "Western Europe", "Europe", "NATO"),
    "Germany":     ("Berlin", "Western Europe", "Europe", "NATO"),
    "Japan":       ("Tokyo", "East Asia", "Asia", "Non-NATO"),
    "South Korea": ("Seoul", "East Asia", "Asia", "Non-NATO"),
    "Pakistan":    ("Islamabad", "South Asia", "Asia", "Non-NATO"),
}


In [11]:
df = pd.read_csv("data/military_metrics_raw.csv")

# Fill mandatory columns using static reference, default "Unknown" if not in dict
df["capital_city"] = df["country"].apply(lambda x: country_meta[x][0] if x in country_meta else "Unknown")
df["region"]       = df["country"].apply(lambda x: country_meta[x][1] if x in country_meta else "Unknown")
df["continent"]    = df["country"].apply(lambda x: country_meta[x][2] if x in country_meta else "Unknown")
df["alliance"]     = df["country"].apply(lambda x: country_meta[x][3] if x in country_meta else "Non-NATO")

df["year"] = datetime.now().year

# Reorder columns
mandatory_cols = ["country", "capital_city", "region", "continent", "alliance", "year", "rank"]
metrics_cols = [c for c in df.columns if c not in mandatory_cols]
df_cleaned = df[mandatory_cols + metrics_cols]

# Save cleaned CSV
df_cleaned.to_csv("data/military_metrics_cleaned.csv", index=False)
df_cleaned.head()


,country,capital_city,region,continent,alliance,year,rank,total-population-by-country,available-military-manpower,manpower-fit-for-military-service,...,natural-gas-production-by-country,natural-gas-consumption-by-country,proven-natural-gas-reserves-by-country,coal-production-by-country,coal-consumption-by-country,proven-coal-reserves-by-country,square-land-area,coastline-coverage,border-coverage,waterway-coverage
0,United States,"Washington, D.C.",North America,North America,NATO,2026,1,3.419634e+08,150463900.0,124816644.0,...,1.029000e+12,9.143010e+11,1.340200e+13,5.342340e+08,4.951560e+08,2.478830e+11,9833517.0,19924.0,12002.0,41009.0
1,Russia,Moscow,Eastern Europe,Europe,Non-NATO,2026,2,1.408208e+08,69002197.0,46189226.0,...,6.178300e+11,4.722390e+11,4.780500e+13,5.311300e+08,2.907630e+08,1.621660e+11,17098242.0,37653.0,22407.0,102000.0
2,China,Beijing,East Asia,Asia,Non-NATO,2026,3,1.415043e+09,764123366.0,626864169.0,...,2.253410e+11,3.661600e+11,6.654000e+12,4.805000e+09,5.191000e+09,1.570410e+11,9596960.0,14500.0,22457.0,27700.0
3,India,New Delhi,South Asia,Asia,Non-NATO,2026,4,1.409128e+09,662290299.0,522786598.0,...,3.317000e+10,5.886700e+10,1.381000e+12,1.020000e+09,1.262000e+09,1.277270e+11,3287263.0,7000.0,13888.0,14500.0
4,South Korea,Seoul,East Asia,Asia,Non-NATO,2026,5,5.208180e+07,26040900.0,21353538.0,...,5.512700e+07,5.948000e+10,7.079000e+09,1.608100e+07,1.368170e+08,3.260000e+08,99720.0,2413.0,237.0,1600.0


## **KPI Feature Engineering**

In [12]:
import pandas as pd
import numpy as np
df = pd.read_csv("data/military_metrics_cleaned.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()


Rows: 145
Columns: 61


,country,capital_city,region,continent,alliance,year,rank,total-population-by-country,available-military-manpower,manpower-fit-for-military-service,...,natural-gas-production-by-country,natural-gas-consumption-by-country,proven-natural-gas-reserves-by-country,coal-production-by-country,coal-consumption-by-country,proven-coal-reserves-by-country,square-land-area,coastline-coverage,border-coverage,waterway-coverage
0,United States,"Washington, D.C.",North America,North America,NATO,2026,1,3.419634e+08,150463900.0,124816644.0,...,1.029000e+12,9.143010e+11,1.340200e+13,5.342340e+08,4.951560e+08,2.478830e+11,9833517.0,19924.0,12002.0,41009.0
1,Russia,Moscow,Eastern Europe,Europe,Non-NATO,2026,2,1.408208e+08,69002197.0,46189226.0,...,6.178300e+11,4.722390e+11,4.780500e+13,5.311300e+08,2.907630e+08,1.621660e+11,17098242.0,37653.0,22407.0,102000.0
2,China,Beijing,East Asia,Asia,Non-NATO,2026,3,1.415043e+09,764123366.0,626864169.0,...,2.253410e+11,3.661600e+11,6.654000e+12,4.805000e+09,5.191000e+09,1.570410e+11,9596960.0,14500.0,22457.0,27700.0
3,India,New Delhi,South Asia,Asia,Non-NATO,2026,4,1.409128e+09,662290299.0,522786598.0,...,3.317000e+10,5.886700e+10,1.381000e+12,1.020000e+09,1.262000e+09,1.277270e+11,3287263.0,7000.0,13888.0,14500.0
4,South Korea,Seoul,East Asia,Asia,Non-NATO,2026,5,5.208180e+07,26040900.0,21353538.0,...,5.512700e+07,5.948000e+10,7.079000e+09,1.608100e+07,1.368170e+08,3.260000e+08,99720.0,2413.0,237.0,1600.0


In [13]:
rename_map = {
    # Population & personnel
    "total-population-by-country": "population",
    "active-military-manpower": "active_personnel",
    "available-military-manpower": "total_personnel",

    # Assets
    "aircraft-total": "total_aircraft",
    "armor-tanks-total": "tanks",
    "navy-ships": "naval_assets",

    # Economy
    "defense-spending-budget": "defense_budget_usd",
    "purchasing-power-parity": "gdp_usd",

    # Geography
    "square-land-area": "land_area_sq_km",
    "coastline-coverage": "coastline_km",

    # Ranking
    "rank": "power_index_rank"
}

df.rename(columns=rename_map, inplace=True)

print("✅ Column standardization completed")
df.columns.tolist()


✅ Column standardization completed


['country',
 'capital_city',
 'region',
 'continent',
 'alliance',
 'year',
 'power_index_rank',
 'population',
 'total_personnel',
 'manpower-fit-for-military-service',
 'manpower-reaching-military-age-annually',
 'active_personnel',
 'active-reserve-military-manpower',
 'manpower-paramilitary',
 'total_aircraft',
 'aircraft-total-fighters',
 'aircraft-total-attack-types',
 'aircraft-total-transports',
 'aircraft-total-trainers',
 'aircraft-total-special-mission',
 'aircraft-total-tanker-fleet',
 'aircraft-helicopters-total',
 'aircraft-helicopters-attack',
 'tanks',
 'armor-apc-total',
 'armor-self-propelled-guns-total',
 'armor-towed-artillery-total',
 'armor-mlrs-total',
 'naval_assets',
 'navy-force-by-tonnage',
 'navy-aircraft-carriers',
 'navy-helo-carriers',
 'navy-submarines',
 'navy-destroyers',
 'navy-frigates',
 'navy-corvettes',
 'navy-patrol-coastal-craft',
 'navy-mine-warfare-craft',
 'defense_budget_usd',
 'external-debt-by-country',
 'gdp_usd',
 'reserves-of-foreign-ex

In [14]:
required_columns = [
    "population",
    "active_personnel",
    "total_personnel",
    "total_aircraft",
    "tanks",
    "naval_assets",
    "defense_budget_usd",
    "gdp_usd",
    "land_area_sq_km",
    "coastline_km",
    "power_index_rank"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"❌ Missing required columns: {missing}")
else:
    print("✅ All KPI-required columns present")


✅ All KPI-required columns present


In [15]:
df["total_assets"] = (
    df["total_aircraft"] +
    df["tanks"] +
    df["naval_assets"]
)


In [18]:
df["assets_per_capita"] = df["total_assets"] / df["population"]


In [19]:
df["budget_to_gdp_ratio"] = df["defense_budget_usd"] / df["gdp_usd"]


In [20]:
df["personnel_density"] = df["total_personnel"] / df["population"]


In [21]:
df["budget_per_soldier"] = df["defense_budget_usd"] / df["active_personnel"]


In [22]:
reference_rank = df["power_index_rank"].min()
df["power_index_rank_gap"] = df["power_index_rank"] - reference_rank


In [23]:
df["air_power_ratio"] = df["total_aircraft"] / df["active_personnel"]


In [24]:
df["armor_intensity"] = df["tanks"] / df["land_area_sq_km"]


In [25]:
df["naval_strength_per_km"] = df["naval_assets"] / df["coastline_km"]


In [27]:
df["military_burden_index"] = df["defense_budget_usd"] / df["population"]


In [28]:
nato_countries = [
    "United States", "United Kingdom", "France", "Germany",
    "Italy", "Canada", "Turkey", "Spain", "Netherlands",
    "Poland", "Norway", "Belgium"
]

coalition_strength = df.loc[
    df["country"].isin(nato_countries),
    "total_assets"
].sum()

df["coalition_strength_index"] = coalition_strength


In [29]:
df.head()


,country,capital_city,region,continent,alliance,year,power_index_rank,population,total_personnel,manpower-fit-for-military-service,...,assets_per_capita,budget_to_gdp_ratio,personnel_density,budget_per_soldier,power_index_rank_gap,air_power_ratio,armor_intensity,naval_strength_per_km,military_burden_index,coalition_strength_index
0,United States,"Washington, D.C.",North America,North America,NATO,2026,1,3.419634e+08,150463900.0,124816644.0,...,0.000053,0.032384,0.44,623766.906971,0,0.009776,0.000474,0.023339,2431.546711,26242.0
1,Russia,Moscow,Eastern Europe,Europe,Non-NATO,2026,2,1.408208e+08,69002197.0,46189226.0,...,0.000075,0.034922,0.49,161089.600000,1,0.003210,0.000329,0.019839,1509.991826,26242.0
2,China,Beijing,East Asia,Asia,Non-NATO,2026,3,1.415043e+09,764123366.0,626864169.0,...,0.000007,0.009018,0.54,148894.348894,2,0.001734,0.000612,0.058000,214.127728,26242.0
3,India,New Delhi,South Asia,Asia,Non-NATO,2026,4,1.409128e+09,662290299.0,522786598.0,...,0.000005,0.007652,0.47,76170.510133,3,0.001526,0.001190,0.049000,77.352786,26242.0
4,South Korea,Seoul,East Asia,Asia,Non-NATO,2026,5,5.208180e+07,26040900.0,21353538.0,...,0.000069,0.017185,0.50,99555.555556,4,0.003422,0.018361,0.089101,860.185340,26242.0


In [30]:
df.to_csv("military_kpi_final.csv", index=False)
print("✅ Final KPI CSV saved as military_kpi_final.csv")


✅ Final KPI CSV saved as military_kpi_final.csv
